In [4]:
# ==========================================
# WATER POTABILITY PREDICTION - RANDOM FOREST
# DATA CLEANING + FEATURE ENGINEERING + SMOTE
# SAVE MODEL + IMPUTER AS PICKLE
# ==========================================

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("C:\\Users\\SAYAN DAS\\OneDrive\\Desktop\\water_app_43\\water_potability.csv")

print("First 5 rows:")
print(df.head())
print("\nShape of dataset:", df.shape)

# =========================
# 2. BASIC CLEANING
# =========================
df = df.drop_duplicates()

print("\nMissing values before cleaning:")
print(df.isnull().sum())

# =========================
# 3. SEPARATE FEATURES AND TARGET
# =========================
X = df.drop("Potability", axis=1)
y = df["Potability"]

# =========================
# 4. HANDLE MISSING VALUES
# =========================
imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# =========================
# 5. FEATURE ENGINEERING
# =========================
X_imputed["ph_Hardness"] = X_imputed["ph"] * X_imputed["Hardness"]
X_imputed["Solids_Conductivity"] = X_imputed["Solids"] * X_imputed["Conductivity"]
X_imputed["Chloramines_Turbidity"] = X_imputed["Chloramines"] * X_imputed["Turbidity"]
X_imputed["Organic_Trihalo"] = X_imputed["Organic_carbon"] * X_imputed["Trihalomethanes"]

print("\nFinal columns used for training:")
print(X_imputed.columns.tolist())
print("\nTotal number of features used for training:", X_imputed.shape[1])

# =========================
# 6. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 7. HANDLE CLASS IMBALANCE
# =========================
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nClass distribution before SMOTE:")
print(y_train.value_counts())

print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# =========================
# 8. BUILD RANDOM FOREST MODEL
# =========================
rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)

# =========================
# 9. PREDICTION
# =========================
y_pred = rf_model.predict(X_test)

# =========================
# 10. EVALUATION
# =========================
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =========================
# 11. FEATURE IMPORTANCE
# =========================
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop Feature Importances:")
print(feature_importance.head(10))

# =========================
# 12. SAVE MODEL AND IMPUTER
# =========================
with open("water_potability_rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("water_potability_imputer.pkl", "wb") as f:
    pickle.dump(imputer, f)

print("\nModel saved as 'water_potability_rf_model.pkl'")
print("Imputer saved as 'water_potability_imputer.pkl'")

First 5 rows:
         ph    Hardness        Solids  Chloramines     Sulfate  Conductivity  \
0       NaN  204.890455  20791.318981     7.300212  368.516441    564.308654   
1  3.716080  129.422921  18630.057858     6.635246         NaN    592.885359   
2  8.099124  224.236259  19909.541732     9.275884         NaN    418.606213   
3  8.316766  214.373394  22018.417441     8.059332  356.886136    363.266516   
4  9.092223  181.101509  17978.986339     6.546600  310.135738    398.410813   

   Organic_carbon  Trihalomethanes  Turbidity  Potability  
0       10.379783        86.990970   2.963135           0  
1       15.180013        56.329076   4.500656           0  
2       16.868637        66.420093   3.055934           0  
3       18.436524       100.341674   4.628771           0  
4       11.558279        31.997993   4.075075           0  

Shape of dataset: (3276, 10)

Missing values before cleaning:
ph                 491
Hardness             0
Solids               0
Chloramines  